# Experiment Tracking & Model Registry

## Learning Objectives
- Understand the importance of experiment tracking
- Implement custom experiment logging
- Track hyperparameters, metrics, and artifacts
- Build a simple model registry
- Compare and reproduce experiments

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import json
import hashlib
from pathlib import Path
from datetime import datetime
from typing import Dict, Any, List, Optional
import joblib

print("Experiment tracking module loaded!")

## 1. Why Track Experiments?

ML development is iterative and involves many experiments. Without tracking:

- You can't reproduce results
- You forget which hyperparameters worked
- You lose track of model performance
- Collaboration becomes difficult

In [ ]:
class ExperimentTracker:
    """Simple experiment tracking system."""
    
    def __init__(self, experiments_dir: str = "experiments"):
        self.experiments_dir = Path(experiments_dir)
        self.experiments_dir.mkdir(exist_ok=True)
        self.current_experiment = None
        self.run_id = None
    
    def start_run(self, experiment_name: str, tags: Dict[str, str] = None):
        """Start a new experiment run."""
        self.run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
        self.current_experiment = {
            "run_id": self.run_id,
            "experiment_name": experiment_name,
            "start_time": datetime.now().isoformat(),
            "tags": tags or {},
            "params": {},
            "metrics": {},
            "artifacts": []
        }
        print(f"Started run: {self.run_id}")
        return self
    
    def log_param(self, key: str, value: Any):
        """Log a hyperparameter."""
        self.current_experiment["params"][key] = value
    
    def log_params(self, params: Dict[str, Any]):
        """Log multiple hyperparameters."""
        self.current_experiment["params"].update(params)
    
    def log_metric(self, key: str, value: float, step: int = None):
        """Log a metric value."""
        if key not in self.current_experiment["metrics"]:
            self.current_experiment["metrics"][key] = []
        self.current_experiment["metrics"][key].append({
            "value": value,
            "step": step,
            "timestamp": datetime.now().isoformat()
        })
    
    def log_artifact(self, artifact_path: str, artifact_type: str = "model"):
        """Log an artifact (model, data, etc.)."""
        self.current_experiment["artifacts"].append({
            "path": str(artifact_path),
            "type": artifact_type,
            "timestamp": datetime.now().isoformat()
        })
    
    def end_run(self, status: str = "COMPLETED"):
        """End the current run and save."""
        self.current_experiment["end_time"] = datetime.now().isoformat()
        self.current_experiment["status"] = status
        
        # Save experiment
        run_dir = self.experiments_dir / self.run_id
        run_dir.mkdir(exist_ok=True)
        
        with open(run_dir / "experiment.json", "w") as f:
            json.dump(self.current_experiment, f, indent=2)
        
        print(f"Run {self.run_id} completed: {status}")
        return self.current_experiment
    
    def list_runs(self) -> List[Dict]:
        """List all experiment runs."""
        runs = []
        for run_dir in sorted(self.experiments_dir.iterdir()):
            exp_file = run_dir / "experiment.json"
            if exp_file.exists():
                with open(exp_file) as f:
                    runs.append(json.load(f))
        return runs


# Demo
tracker = ExperimentTracker("../experiments")
print("ExperimentTracker initialized!")

In [ ]:
# Example: Track an ML experiment
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# Generate data
X, y = make_classification(n_samples=1000, n_features=20, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Hyperparameters to try
params = {
    "n_estimators": 100,
    "max_depth": 10,
    "min_samples_split": 5,
    "random_state": 42
}

# Start experiment
tracker.start_run(
    experiment_name="random_forest_classifier",
    tags={"model_type": "RandomForest", "dataset": "synthetic"}
)

# Log parameters
tracker.log_params(params)
tracker.log_param("train_size", len(X_train))
tracker.log_param("test_size", len(X_test))

# Train model
model = RandomForestClassifier(**params)
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)

# Log metrics
tracker.log_metric("accuracy", accuracy_score(y_test, y_pred))
tracker.log_metric("f1_score", f1_score(y_test, y_pred))
tracker.log_metric("precision", precision_score(y_test, y_pred))
tracker.log_metric("recall", recall_score(y_test, y_pred))

# Save and log artifact
model_path = "../experiments/rf_model.joblib"
joblib.dump(model, model_path)
tracker.log_artifact(model_path, "model")

# End run
result = tracker.end_run()

print("\nExperiment Results:")
print(f"  Accuracy: {result['metrics']['accuracy'][0]['value']:.4f}")
print(f"  F1 Score: {result['metrics']['f1_score'][0]['value']:.4f}")

## 2. Model Registry

A model registry tracks model versions and their lifecycle stages.

In [ ]:
from enum import Enum

class ModelStage(str, Enum):
    DEVELOPMENT = "development"
    STAGING = "staging"
    PRODUCTION = "production"
    ARCHIVED = "archived"


class ModelRegistry:
    """Simple model registry for version management."""
    
    def __init__(self, registry_dir: str = "model_registry"):
        self.registry_dir = Path(registry_dir)
        self.registry_dir.mkdir(exist_ok=True)
        self.registry_file = self.registry_dir / "registry.json"
        self._load_registry()
    
    def _load_registry(self):
        """Load registry from disk."""
        if self.registry_file.exists():
            with open(self.registry_file) as f:
                self.registry = json.load(f)
        else:
            self.registry = {"models": {}}
    
    def _save_registry(self):
        """Save registry to disk."""
        with open(self.registry_file, "w") as f:
            json.dump(self.registry, f, indent=2)
    
    def register_model(
        self,
        name: str,
        version: str,
        model_path: str,
        metrics: Dict[str, float],
        description: str = ""
    ) -> Dict:
        """Register a new model version."""
        if name not in self.registry["models"]:
            self.registry["models"][name] = {"versions": []}
        
        model_entry = {
            "version": version,
            "model_path": str(model_path),
            "metrics": metrics,
            "description": description,
            "stage": ModelStage.DEVELOPMENT.value,
            "registered_at": datetime.now().isoformat(),
            "last_updated": datetime.now().isoformat()
        }
        
        self.registry["models"][name]["versions"].append(model_entry)
        self._save_registry()
        
        print(f"Registered {name} v{version}")
        return model_entry
    
    def transition_stage(self, name: str, version: str, stage: ModelStage):
        """Transition model to new stage."""
        for v in self.registry["models"][name]["versions"]:
            if v["version"] == version:
                v["stage"] = stage.value
                v["last_updated"] = datetime.now().isoformat()
                self._save_registry()
                print(f"Transitioned {name} v{version} to {stage.value}")
                return
    
    def get_latest_version(self, name: str, stage: ModelStage = None) -> Optional[Dict]:
        """Get latest version of a model."""
        if name not in self.registry["models"]:
            return None
        
        versions = self.registry["models"][name]["versions"]
        if stage:
            versions = [v for v in versions if v["stage"] == stage.value]
        
        return versions[-1] if versions else None
    
    def list_models(self) -> List[str]:
        """List all registered models."""
        return list(self.registry["models"].keys())


# Demo
registry = ModelRegistry("../model_registry")

# Register models
registry.register_model(
    name="classifier",
    version="1.0.0",
    model_path="../experiments/rf_model.joblib",
    metrics={"accuracy": 0.92, "f1": 0.91},
    description="Initial RandomForest classifier"
)

# Promote to staging
registry.transition_stage("classifier", "1.0.0", ModelStage.STAGING)

# Get production model
latest = registry.get_latest_version("classifier")
print(f"\nLatest version: {latest['version']} ({latest['stage']})")

## 3. Comparing Experiments

Visualize and compare multiple experiment runs.

In [ ]:
def compare_experiments(tracker: ExperimentTracker) -> pd.DataFrame:
    """Create comparison table of all experiments."""
    runs = tracker.list_runs()
    
    rows = []
    for run in runs:
        row = {
            "run_id": run["run_id"],
            "experiment": run["experiment_name"],
            "status": run.get("status", "UNKNOWN"),
        }
        
        # Add params
        for k, v in run.get("params", {}).items():
            row[f"param_{k}"] = v
        
        # Add metrics (latest value)
        for k, v in run.get("metrics", {}).items():
            if isinstance(v, list) and v:
                row[f"metric_{k}"] = v[-1]["value"]
        
        rows.append(row)
    
    return pd.DataFrame(rows)


# Show comparison
comparison_df = compare_experiments(tracker)
print("Experiment Comparison:")
print(comparison_df.to_string())

## Summary

### Key Takeaways

1. **Track Everything**: Parameters, metrics, artifacts, and metadata
2. **Version Models**: Use semantic versioning (major.minor.patch)
3. **Stage Transitions**: Development → Staging → Production
4. **Compare Experiments**: Make data-driven decisions
5. **Reproducibility**: Store enough info to reproduce any experiment

In [ ]:
print("=" * 50)
print("EXPERIMENT TRACKING COMPLETE")
print("=" * 50)
print("\nKey Components:")
print("  📊 ExperimentTracker - Log params, metrics, artifacts")
print("  📦 ModelRegistry - Version and stage management")
print("  📈 Comparison - Compare across experiments")